In [1]:
pip install dash plotly numpy

   ---------------------------------------- 0.0/7.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/7.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/7.2 MB ? eta -:--:--
   ------- -------------------------------- 1.3/7.2 MB 5.6 MB/s eta 0:00:02
   ----------------- ---------------------- 3.1/7.2 MB 7.0 MB/s eta 0:00:01
   ---------------------------- ----------- 5.2/7.2 MB 7.9 MB/s eta 0:00:01
   ---------------------------------------  7.1/7.2 MB 8.5 MB/s eta 0:00:01
   ---------------------------------------- 7.2/7.2 MB 8.0 MB/s  0:00:01

   -------------------- ------------------- 1/2 [dash]
   -------------------- ------------------- 1/2 [dash]
   -------------------- ------------------- 1/2 [dash]
   -------------------- ------------------- 1/2 [dash]
   -------------------- ------------------- 1/2 [dash]
   -------------------- ------------------- 1/2 [dash]
   -------------------- ------------------- 1/2 [dash]
   -------------------- ------

In [1]:
import dash
from dash import dcc, html
import plotly.graph_objs as go
import numpy as np

# Initialize app
app = dash.Dash(__name__)

# ---- Sample simulation data ----
time = np.linspace(0, 500, 200)

efficiency = np.piecewise(
    time,
    [time < 450, time >= 450],
    [lambda t: 0, lambda t: 70 + 3 * np.sin(0.2 * t)]
)

waste = 100 - efficiency

# ---- Layout ----
app.layout = html.Div([

    html.H2(
        "Hospital Operations Analytics & Simulation",
        style={'textAlign': 'center'}
    ),

    # ---- KPI Section ----
    html.Div([
        html.H4("Hospital Efficiency Dashboard"),

        html.Div([
            html.Div([
                html.P("EFFICIENCY"),
                html.H3(f"{np.mean(efficiency):.1f}%"),
            ], style={'width': '30%', 'textAlign': 'center'}),

            html.Div([
                html.P("RESOURCE WASTE"),
                html.H3(f"{np.mean(waste):.1f}%"),
            ], style={'width': '30%', 'textAlign': 'center'}),

            html.Div([
                html.P("DATA ACCURACY"),
                html.H3("96.4%"),
            ], style={'width': '30%', 'textAlign': 'center'}),
        ], style={
            'display': 'flex',
            'justifyContent': 'space-around'
        })
    ], style={'marginBottom': '20px'}),

    # ---- Process Flow ----
    html.Div([
        html.Div("TRIAGE", style={'width': '30%', 'textAlign': 'center'}),

        html.Div([
            html.Div("IDENTIFIED", style={
                'border': '2px solid green',
                'borderRadius': '10px',
                'padding': '5px',
                'display': 'inline-block'
            }),
            html.Div("ANALYSIS", style={'marginTop': '10px'})
        ], style={'width': '30%', 'textAlign': 'center'}),

        html.Div("OUTCOME", style={'width': '30%', 'textAlign': 'center'}),
    ], style={
        'display': 'flex',
        'justifyContent': 'space-around',
        'marginBottom': '40px'
    }),

    # ---- Graph ----
    dcc.Graph(
        id='performance-graph',
        figure={
            'data': [
                go.Scatter(
                    x=time,
                    y=efficiency,
                    mode='lines',
                    name='Efficiency'
                ),
                go.Scatter(
                    x=time,
                    y=waste,
                    mode='lines',
                    name='Waste',
                    line=dict(dash='dash')
                ),
            ],
            'layout': go.Layout(
                xaxis={'title': 'Simulation Time (s)'},
                yaxis={'title': 'Score (%)', 'range': [0, 100]},
                hovermode='closest'
            )
        }
    )
])

# ---- Run server ----
if __name__ == '__main__':
    app.run(debug=True)

In [4]:
import dash
from dash import dcc, html
from dash.dependencies import Input, Output
import pandas as pd
import plotly.graph_objs as go

app = dash.Dash(__name__)

# ---- Load Dataset ----
df = pd.read_csv("C:/Users/saidh/Downloads/FACT_INSURANCE_CLAIM.csv")

# ---- Layout ----
app.layout = html.Div([

    html.H2("Hospital Claims & Financial Live Dashboard",
            style={'textAlign': 'center'}),

    # ---- KPIs ----
    html.Div([

        html.Div([
            html.P("Claim Approval Rate"),
            html.H3(id='claim-kpi')
        ], style={'width': '30%', 'textAlign': 'center'}),

        html.Div([
            html.P("Rejected Claims"),
            html.H3(id='reject-kpi')
        ], style={'width': '30%', 'textAlign': 'center'}),

        html.Div([
            html.P("Total Cash Flow (₹)"),
            html.H3(id='finance-kpi')
        ], style={'width': '30%', 'textAlign': 'center'}),

    ], style={'display': 'flex', 'justifyContent': 'space-around'}),

    # ---- Graph ----
    dcc.Graph(id='live-graph'),

    # ---- Interval ----
    dcc.Interval(
        id='interval',
        interval=2000,  # 2 seconds
        n_intervals=0
    )
])


# ---- Callback ----
@app.callback(
    [Output('live-graph', 'figure'),
     Output('claim-kpi', 'children'),
     Output('reject-kpi', 'children'),
     Output('finance-kpi', 'children')],
    [Input('interval', 'n_intervals')]
)
def update(n):

    # Create live effect (gradually increasing data)
    data = df.iloc[:min(n + 20, len(df))]

    total = len(data)

    approved = (data['claim_status'] == 'Approved').sum()
    rejected = (data['claim_status'] == 'Rejected').sum()

    approval_rate = (approved / total) * 100 if total > 0 else 0
    reject_rate = (rejected / total) * 100 if total > 0 else 0

    cash_flow = data['claim_amount'].sum()

    # ---- Graph ----
    fig = {
        'data': [
            go.Scatter(
                x=list(range(len(data))),
                y=data['claim_amount'],
                mode='lines+markers',
                name='Claim Amount (₹)'
            )
        ],
        'layout': go.Layout(
            title="Claim Amount Trend",
            xaxis={'title': 'Number of Claims'},
            yaxis={'title': 'Amount (₹)'}
        )
    }

    # ✅ Correct return order
    return (
        fig,
        f"{approval_rate:.1f}%",
        f"{reject_rate:.1f}%",
        f"₹ {int(cash_flow)}"
    )


# ---- Run ----
if __name__ == '__main__':
    app.run(debug=True)

[2026-10-29 14:02:25,760] ERROR in app: Exception on /_dash-update-component [POST]
Traceback (most recent call last):
  File "C:\ProgramData\anaconda3\Lib\site-packages\dash\dash.py", line 1493, in _prepare_callback
    cb = self.callback_map[output]
         ~~~~~~~~~~~~~~~~~^^^^^^^^
KeyError: '..live-graph.figure...claim-kpi.children...bed-kpi.children...finance-kpi.children..'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "C:\ProgramData\anaconda3\Lib\site-packages\flask\app.py", line 917, in full_dispatch_request
    rv = self.dispatch_request()
  File "C:\ProgramData\anaconda3\Lib\site-packages\flask\app.py", line 902, in dispatch_request
    return self.ensure_sync(self.view_functions[rule.endpoint])(**view_args)  # type: ignore[no-any-return]
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^
  File "C:\ProgramData\anaconda3\Lib\site-packages\dash\_get_app.py", line 17, in wrap
    retu